In [ ]:
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

def MNIST_loader():
    training_data = datasets.MNIST('winter/deep_compression/data/MNIST/',
                                 download=True,
                                 train=True,transform=ToTensor())
    test_data = datasets.MNIST('winter/deep_compression/data/MNIST/',
                                    download=True,
                                    train=False,transform=ToTensor())
    train_dataloader = DataLoader(training_data , batch_size = 32 , shuffle = True)
    test_dataloader = DataLoader(test_data , batch_size = 32 , shuffle = True)

    return train_dataloader,test_dataloader


def CIFAR10_loader():
    training_data = datasets.CIFAR10('winter/deep_compression/data/CIFAR10/',
                                 download=True,
                                 train=True,transform=ToTensor())
    test_data = datasets.CIFAR10('winter/deep_compression/data/CIFAR10/',
                                    download=True,
                                    train=False,transform=ToTensor())
    train_dataloader = DataLoader(training_data , batch_size = 32 , shuffle = True)
    test_dataloader = DataLoader(test_data , batch_size = 32 , shuffle = True)

    return train_dataloader,test_dataloader


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.cluster import KMeans

class modified_linear(nn.Linear):
    def __init__(self, in_features, out_features,mode = 'normal', bias = True, device=None, dtype=None):
        super().__init__(in_features, out_features, bias, device, dtype)
        self.mode = mode
        self.register_buffer("mask", torch.ones_like(self.weight))
        self.register_buffer(
            "assignments",
            torch.zeros_like(self.weight, dtype=torch.long)  
        )
        self.hashtable = None 
    
    def prune(self,threshold):
        self.mode = 'prune'
        new_mask = (self.weight.abs() > threshold).float()   
        self.mask.mul_(new_mask)
        print("pruned!!")
        return
    
    def quantize(self,k):
        self.mode = 'quantize'
        matrix = self.weight.detach().cpu().numpy()
        kmeans = KMeans(
                n_clusters=k,
                random_state=42,
                n_init=10
            )
        kmeans.fit(matrix.reshape(-1,1))
        assignments =torch.from_numpy(kmeans.labels_).reshape(self.weight.shape).long()
        self.assignments.copy_(assignments)
        self.hashtable = nn.Parameter(
            torch.from_numpy(
                kmeans.cluster_centers_.reshape(-1)
            ).to(self.weight.device, self.weight.dtype)
        )
        self.weight.requires_grad = False
        return
    
    def forward(self, input):
        if(self.mode == 'normal'):
            W = self.weight 
        elif(self.mode == 'prune'):
            W = (self.weight * self.mask)
        elif(self.mode == 'quantize'):
            W = self.hashtable[self.assignments] * self.mask  
        return F.linear(input,W,self.bias)

class modified_conv2d(nn.Conv2d):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        padding=0,
        dilation=1,
        groups=1,
        bias=True,
        padding_mode="zeros",
        mode="normal",
        device=None,
        dtype=None,
    ):
        super().__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride,
            padding,
            dilation,
            groups,
            bias,
            padding_mode,
            device=device,
            dtype=dtype,
        )
        self.mode = mode
        self.register_buffer("mask", torch.ones_like(self.weight))
        self.register_buffer(
            "assignments",
            torch.zeros_like(self.weight, dtype=torch.long),
        )
        self.hashtable = None 

    def prune(self, threshold):
        self.mode = "prune"
        new_mask = (self.weight.abs() > threshold).float()
        self.mask.mul_(new_mask)  
        print("pruned!!")
        return

    def quantize(self, k):
        self.mode = "quantize"
        matrix = self.weight.detach().cpu().numpy()
        kmeans = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10,
        )
        kmeans.fit(matrix.reshape(-1, 1))
        assignments = torch.from_numpy(
            kmeans.labels_
        ).reshape(self.weight.shape).long()
        self.assignments.copy_(assignments)
        self.hashtable = nn.Parameter(
            torch.from_numpy(
                kmeans.cluster_centers_.reshape(-1)
            ).to(self.weight.device, self.weight.dtype)
        )
        self.weight.requires_grad = False
        return

    def forward(self, input):
        if self.mode == "normal":
            W = self.weight
        elif self.mode == "prune":
            W = self.weight * self.mask
        elif self.mode == "quantize":
            W = self.hashtable[self.assignments] * self.mask
        return F.conv2d(
            input,
            W,
            self.bias,
            self.stride,
            self.padding,
            self.dilation,
            self.groups,
        )

In [ ]:
from torch import nn

class mnist_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            modified_linear(28 * 28, 256),
            nn.ReLU(),
            modified_linear(256, 512),
            nn.ReLU(),
            modified_linear(512, 256),
            nn.ReLU(),
            modified_linear(256, 10)
        )
        
    def forward(self,x):
        x = self.flatten(x)
        x = self.classifier(x)
        return x
    
    def prune(self,threshold):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.prune(threshold)
    
    def quantize(self,k):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.quantize(k)

class SmallCIFARNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            modified_conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            modified_conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), 

            modified_conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            modified_conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  
            
            modified_conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), 
        )
        self.flatten = nn.Flatten()
        self.softmax = nn.Softmax(dim = 1)
        self.classifier = nn.Sequential(
            modified_linear(128 * 4 * 4, 256),
            nn.ReLU(),
            modified_linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

    def prune(self,threshold):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.prune(threshold)
    
    def quantize(self,k):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.quantize(k)

In [ ]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

In [ ]:
import torch.optim as optim
from torch import nn
from tqdm import tqdm
import torch
def train_model(model, train_loader,device,epochs=5,lr=0.01, mode="Training"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss/len(train_loader):.4f}")

def evaluate(model, loader , device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total


In [ ]:
import torch
from torch import nn

def prune_model(model, prune_fraction):
    assert 0.0 < prune_fraction < 1.0
    all_weights = []
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w = module.weight.detach().abs().flatten()
            all_weights.append(w)
    all_weights = torch.cat(all_weights)

    k = int(prune_fraction * all_weights.numel())
    threshold = torch.kthvalue(all_weights, k).values.item()
    model.prune(threshold)
    return 

def quantize_model(model,K):
    model.quantize(K)
    return

In [ ]:
model = SmallCIFARNet()
train_dataloader ,test_dataloader = CIFAR10_loader()
model.to(device)
train_model(model,train_dataloader,device,5)
print(evaluate(model,test_dataloader,device))
prune_model(model,0.9)
print(evaluate(model,test_dataloader,device))
train_model(model,train_dataloader,device,5)
print(evaluate(model,test_dataloader,device))
quantize_model(model,32)
print(evaluate(model,test_dataloader,device))
train_model(model,train_dataloader,device,5)
print(evaluate(model,test_dataloader,device))



In [ ]:
import numpy as np
def is_compressed_layer(module):
    return isinstance(module, (modified_linear, modified_conv2d))

def export_compressed_model(model):
    """
    Returns:
        dict: layer_name -> compressed data
    """
    compressed = {}

    for name, module in model.named_modules():
        if is_compressed_layer(module):

            layer_data = {}

            # shape (for sanity / reconstruction)
            layer_data["shape"] = tuple(module.weight.shape)

            # mask (bool)
            layer_data["mask"] = (
                module.mask.detach()
                .cpu()
                .numpy()
                .astype(np.bool_)
            )

            # assignments (cluster indices)
            layer_data["assignments"] = (
                module.assignments.detach()
                .cpu()
                .numpy()
                .astype(np.uint8)
            )

            # codebook (centroids)
            layer_data["codebook"] = (
                module.hashtable.detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            # bias (optional)
            if module.bias is not None:
                layer_data["bias"] = (
                    module.bias.detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )
            else:
                layer_data["bias"] = None

            compressed[name] = layer_data

    return compressed

def save_model_npz(model, savepath):
    compressed = export_compressed_model(model)
    np.savez(savepath, **compressed)

import torch

def load_model_npz(model, npz_path, device="cpu"):
    data = np.load(npz_path, allow_pickle=True)

    for name, module in model.named_modules():
        if is_compressed_layer(module) and name in data:

            layer_data = data[name].item()

            # restore mask
            module.mask.copy_(
                torch.from_numpy(layer_data["mask"])
                .to(device)
                .float()
            )

            # restore assignments
            module.assignments.copy_(
                torch.from_numpy(layer_data["assignments"])
                .to(device)
                .long()
            )

            # restore codebook
            module.hashtable = torch.nn.Parameter(
                torch.from_numpy(layer_data["codebook"])
                .to(device)
                .float()
            )

            # restore bias
            if layer_data["bias"] is not None and module.bias is not None:
                module.bias.data.copy_(
                    torch.from_numpy(layer_data["bias"])
                    .to(device)
                )

            # neutralize dense weights
            module.weight.data.zero_()
            module.weight.requires_grad = False

            # force compressed execution
            module.mode = "quantize"

    return model

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

def load_csr_from_npz(npz_path):
    raw = np.load(npz_path, allow_pickle=True)
    csr_layers = {}

    for layer_name in raw.files:
        layer = raw[layer_name].item()

        mask = layer["mask"]             
        assignments = layer["assignments"] 
        shape = tuple(layer["shape"])
        codebook = layer["codebook"]
        bias = layer["bias"]

        rows, cols = np.nonzero(mask)

        values = assignments[rows, cols]

        csr = csr_matrix(
            (values, (rows, cols)),
            shape=shape
        )

        csr_layers[layer_name] = {
            "csr": csr,
            "codebook": codebook,
            "bias": bias,
            "shape": shape
        }

    return csr_layers
csr_layers = load_csr_from_npz("compressed_model.npz")

layer0 = csr_layers["classifier.0"]
print(layer0["csr"])
print(layer0["codebook"])